# Ames Housing: What relates to sale price?

**Question:** How strongly are observable home size and overall quality associated with sale price, and can these features predict prices on unseen rows? This notebook provides descriptive analysis and a train/test-evaluated bonus model.

Source and terms: Kaggle House Prices competition; see the project README. This is historical Ames data, not a current valuation dataset.

**GitHub repository:** https://github.com/roee0910/cs82a-portfolio/tree/main/final_project

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import SVG, display

DATA = Path("data/train.csv")
df = pd.read_csv(DATA)
print("Rows, columns:", df.shape)
print("Missing cells:", int(df.isna().sum().sum()))
df.head()

## Cleaning and descriptive statistics
`Id` is excluded. Categorical blanks are retained as a Missing level; numeric missing values are imputed from each training partition only. We do not remove unusual sale prices.

In [ ]:
print(df[["SalePrice", "OverallQual", "GrLivArea"]].describe().round(1))
print("Median sale price:", df.SalePrice.median())
print("Sale price / living area Pearson correlation:", df[["SalePrice", "GrLivArea"]].corr().iloc[0,1])
print("Top neighborhood medians:")
print(df.groupby("Neighborhood").SalePrice.agg(["median", "count"]).sort_values("median", ascending=False).head(5).round(0))

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

In [ ]:
# Recreate the three labeled SVG charts from the loaded dataset.
from pathlib import Path
FIG = Path("figures"); FIG.mkdir(exist_ok=True)
def save_bars(labels, values, title, ylabel, path):
    w,h=900,520; left,bottom,top=80,440,80; mx=max(values)*1.1
    s=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">{title}</text>',f'<line x1="{left}" y1="{top}" x2="{left}" y2="{bottom}" stroke="#333"/><line x1="{left}" y1="{bottom}" x2="850" y2="{bottom}" stroke="#333"/><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">{ylabel}</text>']
    bw=770/len(values)*.65
    for i,(label,value) in enumerate(zip(labels,values)):
        x=left+(i+.5)*770/len(values)-bw/2; yy=bottom-value/mx*(bottom-top)
        s += [f'<rect x="{x:.1f}" y="{yy:.1f}" width="{bw:.1f}" height="{bottom-yy:.1f}" fill="#2673a8"/>',f'<text x="{x+bw/2:.1f}" y="{bottom+22}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>',f'<text x="{x+bw/2:.1f}" y="{yy-7:.1f}" text-anchor="middle" font-family="Arial" font-size="11">${value:,.0f}</text>']
    Path(path).write_text(''.join(s)+'</svg>')
def save_scatter(x,y,path):
    s=['<svg xmlns="http://www.w3.org/2000/svg" width="900" height="520"><rect width="100%" height="100%" fill="white"/><text x="450" y="35" text-anchor="middle" font-family="Arial" font-size="22" font-weight="bold">Living area and sale price</text><line x1="80" y1="80" x2="80" y2="440" stroke="#333"/><line x1="80" y1="440" x2="850" y2="440" stroke="#333"/><text x="465" y="495" text-anchor="middle" font-family="Arial" font-size="15">Above-ground living area (sq ft)</text><text x="25" y="260" transform="rotate(-90 25,260)" font-family="Arial" font-size="15">Sale price (USD)</text>']
    for xx,yy in zip(x,y): s.append(f'<circle cx="{80+xx/6000*770:.1f}" cy="{440-yy/800000*360:.1f}" r="2.2" fill="#2878a5" fill-opacity=".4"/>')
    Path(path).write_text(''.join(s)+'</svg>')
quality = df.groupby("OverallQual").SalePrice.median()
nb_medians = df.groupby("Neighborhood").SalePrice.median().sort_values(ascending=False).head(5)
save_bars(list(map(str,quality.index)), quality.values, "Sale price rises with overall quality", "Median sale price (USD)", FIG/"quality.svg")
save_bars(nb_medians.index.tolist(), nb_medians.values, "Five highest neighborhood medians", "Median sale price (USD)", FIG/"neighborhoods.svg")
save_scatter(df.GrLivArea.to_numpy(), df.SalePrice.to_numpy(), FIG/"area_price.svg")
print("Generated 3 charts in", FIG)

## Chart 1 — quality and price
Each bar is a group median; this is descriptive and does not imply a causal effect.

In [ ]:
display(SVG(filename="figures/quality.svg"))

## Chart 2 — neighborhood and price
These are the five highest observed neighborhood medians. Group sizes differ, so interpret cautiously.

In [ ]:
display(SVG(filename="figures/neighborhoods.svg"))

## Chart 3 — living area and price
The point cloud shows both the positive association and the considerable variation left unexplained by size alone.

In [ ]:
display(SVG(filename="figures/area_price.svg"))

## Bonus: baseline, Ridge, Lasso, and Random Forest

We predict natural-log sale price, `log(SalePrice)`. The data are split once into 80% train and 20% test with `random_state=42`. Five-fold shuffled cross-validation runs **only within train**. A `Pipeline` contains imputation, scaling/one-hot encoding, and estimator, so each fold learns preprocessing only from that fold. We choose the model family by mean CV log-RMSE, then evaluate each frozen candidate once on the test set for comparison; test metrics do not choose the winner.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.base import clone
from sklearn.metrics import root_mean_squared_error, r2_score

X = df.drop(columns=["SalePrice", "Id"])
y = np.log(df["SalePrice"].astype(float))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
numeric = X_train.select_dtypes(include="number").columns.tolist()
categorical = [c for c in X_train.columns if c not in numeric]
preprocess = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                              ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
models = {
    "Mean baseline": Pipeline([("preprocess", preprocess), ("model", DummyRegressor(strategy="mean"))]),
    "Ridge": Pipeline([("preprocess", preprocess), ("model", Ridge(alpha=10.0))]),
    "Lasso": Pipeline([("preprocess", preprocess), ("model", Lasso(alpha=0.0005, max_iter=20000))]),
    "Random Forest": Pipeline([("preprocess", preprocess), ("model", RandomForestRegressor(
        n_estimators=300, max_features=0.7, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=cv,
        scoring={"rmse": "neg_root_mean_squared_error", "r2": "r2"},
        n_jobs=1, return_train_score=False)
    rows.append({"Model": name,
        "CV RMSE log mean": -scores["test_rmse"].mean(),
        "CV RMSE log SD": scores["test_rmse"].std(ddof=1),
        "CV R² mean": scores["test_r2"].mean(),
        "CV R² SD": scores["test_r2"].std(ddof=1)})
cv_results = pd.DataFrame(rows).sort_values("CV RMSE log mean")
best_name = cv_results.iloc[0]["Model"]
print("CV model selection (lower RMSE is better):")
display(cv_results.round(4))
print("Selected using train-only CV:", best_name)

In [ ]:
# Fit on the complete training split and evaluate on the untouched test split.
# Evaluate all candidates for a transparent comparison; the CV winner remains preselected.
fits, test_rows = {}, []
for name, estimator in models.items():
    fitted = clone(estimator).fit(X_train, y_train)
    fits[name] = fitted
    pred = fitted.predict(X_test)
    test_rows.append({"Model": name,
        "Test RMSE log": root_mean_squared_error(y_test, pred),
        "Test R²": r2_score(y_test, pred)})
test_results = pd.DataFrame(test_rows)
comparison = cv_results.merge(test_results, on="Model")
display(comparison.round(4))

selected_model = fits[best_name]
y_pred = selected_model.predict(X_test)
residuals = y_test.to_numpy() - y_pred
print(f"Selected: {best_name}; test log-RMSE={root_mean_squared_error(y_test, y_pred):.4f}; "
      f"test R²={r2_score(y_test, y_pred):.4f}")
print("Residual mean / median:", round(residuals.mean(), 4), "/", round(np.median(residuals), 4))
print("Residual 5th / 95th percentiles:", np.round(np.quantile(residuals, [0.05, 0.95]), 4))
print("Correlation(prediction, residual):", round(np.corrcoef(y_pred, residuals)[0,1], 3))

### Residual check
Residual is `observed log(price) − predicted log(price)`. The diagnostic below checks for a broad trend against fitted values and shows the error distribution. It does not replace testing on new data.

In [ ]:
# Simple SVG residual plot, generated without extra plotting dependencies.
from pathlib import Path
def residual_svg(pred, resid, output):
    w,h=900,500; x0,x1=float(min(pred)-.05),float(max(pred)+.05); r=max(abs(resid))*1.15
    parts=[f'<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}"><rect width="100%" height="100%" fill="white"/><text x="450" y="32" text-anchor="middle" font-family="Arial" font-size="21" font-weight="bold">Lasso test residuals versus fitted log price</text>', '<line x1="80" y1="250" x2="850" y2="250" stroke="#555" stroke-dasharray="6,5"/>', '<line x1="80" y1="70" x2="80" y2="430" stroke="#333"/><line x1="80" y1="430" x2="850" y2="430" stroke="#333"/>', '<text x="465" y="480" text-anchor="middle" font-family="Arial" font-size="15">Predicted log(price)</text>', '<text x="24" y="260" transform="rotate(-90 24,260)" font-family="Arial" font-size="15">Observed − predicted log(price)</text>']
    for x,yv in zip(pred,resid):
        cx=80+(x-x0)/(x1-x0)*770; cy=250-yv/r*180
        parts.append(f'<circle cx="{cx:.1f}" cy="{cy:.1f}" r="3" fill="#2878a5" fill-opacity=".5"/>')
    Path(output).write_text(''.join(parts)+'</svg>')
residual_svg(y_pred, residuals, "figures/lasso_test_residuals.svg")
display(SVG(filename="figures/lasso_test_residuals.svg"))

## Interpretation and limitations
Use the CV comparison to select the model; report the held-out test scores as final performance estimates. Log-RMSE is in natural-log units and R² is unitless. These scores describe a random holdout from historical Ames sales, not a future market. The data are one city, primarily 2006–2010; associations are not causal and individual predictions are not financial advice. See README for the full report, source/license note, cleaning log, and reproducibility steps.